# CardWatch terminal day analysis

**Purpose.** Ingest the synthetic branch and security logs, build one row per POS terminal per calendar day, establish clean pre-incident baselines, detect deviations, and reconstruct an incident timeline. Run the cells top to bottom. The final CSV exports go to `cardwatch_outputs`. Synthetic outcomes illustrate methods; they are not real-world detection performance.

**Scope.** The 14 days from 13 to 26 June 2026 are reserved for baselining; the 90-day study period begins 27 June and ends 24 September inclusive. `ground_truth_private.csv` is used only for supervised training labels and a separate final test, never as a feature. Dispute flags are used only for retrospective investigation. The six infections are injected, and their obvious signatures can inflate metrics.

## 1. Generator, Imports and input folder

Upload the CSV files into one Colab folder, or mount Google Drive and change `DATA_DIR` to your folder. In local execution the notebook finds the adjacent `cardwatch_data` folder. The library versions of the CSVs are individually downloadable.

In [43]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ipaddress, math, re
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value)
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/CardWatch/data')

import subprocess
import sys
from pathlib import Path

generator = Path(
    "/content/drive/MyDrive/CardWatch/generate_cardwatch.py"
)

if not generator.exists():
    raise FileNotFoundError(
        "Upload generate_cardwatch.py into My Drive/CardWatch first."
    )

subprocess.run(
    [
        sys.executable,
        str(generator),
        "--out",
        "/content/drive/MyDrive/CardWatch/data",
        "--seed",
        "82119",
    ],
    check=True,
)

print("All input files regenerated as one consistent dataset.")

print("Files found:")
for file in sorted(DATA_DIR.glob("*")):
    print(file.name)

if not (DATA_DIR / 'branches.csv').exists():
    raise FileNotFoundError(
        f"branches.csv was not found in {DATA_DIR}. "
        "Check the folder location and filename."
    )

CANDIDATES = [Path('/content/cardwatch_data'), Path('/content/drive/MyDrive/CardWatch/data'), Path('cardwatch_data')]
DATA_DIR = next((p for p in CANDIDATES if (p / 'branches.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Upload the CSV files into /content/cardwatch_data or set DATA_DIR to your Google Drive folder.')
OUT = Path('/content/drive/MyDrive/CardWatch/outputs')
OUT.mkdir(parents=True, exist_ok=True)
BASE_START = pd.Timestamp('2026-06-13')
ANALYSIS_START = pd.Timestamp('2026-06-27')
END_EXCLUSIVE = pd.Timestamp('2026-09-25')
print('Input:', DATA_DIR.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All input files regenerated as one consistent dataset.
Files found:
README.md
authentication_logs.csv
branches.csv
devices.csv
dns_logs.csv
edr_events.csv
ground_truth_private.csv
inventory_exceptions.csv
manifest.json
network_flows.csv
payment_auth.csv
pos_events.csv
shifts.csv
soc_tickets.csv
system_logs.csv
Input: /content/drive/MyDrive/CardWatch/data


## 2. Load and validate the sources

The loader converts timestamps during ingestion and leaves identifiers as strings. It checks and verifies that the device inventory has unique IDs, that every POS belongs to a branch, and that sales join to payment authorisations using `txn_ref`. Network and DNS fields refer to the same terminal via their device ID.

In [44]:
sources = {
    'branches':'branches.csv','devices':'devices.csv','shifts':'shifts.csv',
    'pos':'pos_events.csv','payments':'payment_auth.csv',
    'exceptions':'inventory_exceptions.csv','edr':'edr_events.csv',
    'flows':'network_flows.csv','dns':'dns_logs.csv',
    'auth':'authentication_logs.csv','system':'system_logs.csv',
    'tickets':'soc_tickets.csv'
}
required = list(sources.values()) + ['ground_truth_private.csv']
missing = [name for name in required if not (DATA_DIR/name).exists()]
if missing: raise FileNotFoundError(f'Missing files: {missing}')
raw = {key:pd.read_csv(DATA_DIR/file, low_memory=False, dtype={'txn_ref':'string'}) for key,file in sources.items()}
for key,col in {'pos':'timestamp','payments':'auth_timestamp','exceptions':'timestamp',
                'edr':'timestamp','flows':'timestamp','dns':'timestamp',
                'auth':'timestamp','system':'timestamp','tickets':'opened_at',
                'shifts':'shift_start'}.items():
    raw[key][col] = pd.to_datetime(raw[key][col], errors='raise')
raw['shifts']['shift_end'] = pd.to_datetime(raw['shifts']['shift_end'])
branches,devices = raw['branches'],raw['devices']
pos_devices = devices.query("device_type == 'POS' and analysis_scope == 1").copy()
assert branches.branch_id.is_unique and devices.device_id.is_unique
assert len(branches)==42 and len(pos_devices)==54
assert set(pos_devices.branch_id).issubset(set(branches.branch_id))
sales = raw['pos'].query("event_type == 'sale'")
assert sales.txn_ref.notna().all() and sales.txn_ref.is_unique
assert raw['payments'].txn_ref.is_unique
assert set(sales.txn_ref) == set(raw['payments'].txn_ref)
print({key:len(frame) for key,frame in raw.items()})
print('Payment join coverage:', len(set(sales.txn_ref) & set(raw['payments'].txn_ref))/len(sales))

{'branches': 42, 'devices': 252, 'shifts': 5616, 'pos': 54250, 'payments': 53344, 'exceptions': 906, 'edr': 168, 'flows': 31012, 'dns': 29376, 'auth': 5723, 'system': 134, 'tickets': 6}
Payment join coverage: 1.0


## 3. Build the terminal day spine

Create every terminal and date combination first, including days that are more quiet. Aggregated logs then joined to this. This avoids silently excluding terminal days with no recorded activity. The join key used is (`terminal_id`, `date`).

In [45]:
days = pd.date_range(BASE_START, END_EXCLUSIVE-pd.Timedelta(days=1), freq='D')
spine = pd.MultiIndex.from_product([pos_devices.device_id,days], names=['terminal_id','date']).to_frame(index=False)
spine = spine.merge(pos_devices[['device_id','branch_id','ip_address']], left_on='terminal_id',right_on='device_id',validate='many_to_one').drop(columns='device_id')
spine = spine.merge(branches[['branch_id','region','profile','trading_open','trading_close']],on='branch_id',validate='many_to_one')
assert len(spine)==54*104
print('Terminal-day records:', len(spine))

Terminal-day records: 5616


## 4. Aggregate POS and payment activity

Count approved sales and exceptions for each terminal-day. The payment table supplies transaction context for investigation; its dispute flags stay out of detection features because they are generally known later than the transaction.

In [46]:
pos = raw['pos'].copy(); pos['date']=pos.timestamp.dt.normalize()
pos_agg = pos.groupby(['terminal_id','date']).agg(
    sales_count=('event_type', lambda s:(s=='sale').sum()),
    exception_count=('event_type', lambda s:s.isin(['void','refund','no_sale']).sum()),
    override_count=('supervisor_override','sum'),
    total_amount=('amount',lambda s:pd.to_numeric(s,errors='coerce').fillna(0).sum())
).reset_index()
features=spine.merge(pos_agg,on=['terminal_id','date'],how='left',validate='one_to_one')
assert len(features)==len(spine)

## 5. Network flows and DNS

For each terminal-day, calculate external destination count, outbound bytes, outbound/inbound asymmetry, after-hours external sessions, repeated-destination concentration, DNS length and TXT query count. These signals reflect the charter’s beacon and tunnelling questions. RFC documentation addresses are treated as external in this synthetic dataset.

In [47]:
flows=raw['flows'].copy(); flows['date']=flows.timestamp.dt.normalize()
for c in ['bytes_out','bytes_in','duration_seconds']:
    flows[c]=pd.to_numeric(flows[c],errors='coerce').fillna(0)
flows['external']=~flows.dst_ip.fillna('').str.startswith(('10.','172.16.','192.168.'))
flows['hour']=flows.timestamp.dt.hour
flows=flows.merge(branches[['branch_id','trading_open','trading_close']],on='branch_id',validate='many_to_one')
flows['after_hours']=(flows.hour<flows.trading_open.str[:2].astype(int)) | (flows.hour>=flows.trading_close.str[:2].astype(int))
extflows=flows.loc[flows.external].copy()
net_agg=extflows.groupby(['src_device_id','date']).agg(
    external_flows=('dst_ip','size'),distinct_external_ips=('dst_ip','nunique'),
    bytes_out=('bytes_out','sum'),bytes_in=('bytes_in','sum'),
    max_flow_out=('bytes_out','max'),off_hours_flows=('after_hours','sum'),
    long_sessions=('duration_seconds',lambda s:(s>=90).sum())
).reset_index().rename(columns={'src_device_id':'terminal_id'})
net_agg['out_in_ratio']=net_agg.bytes_out/(net_agg.bytes_in+1)
features=features.merge(net_agg,on=['terminal_id','date'],how='left',validate='one_to_one')

dns=raw['dns'].copy(); dns['date']=dns.timestamp.dt.normalize()
dns['query_length']=pd.to_numeric(dns.query_length,errors='coerce').fillna(0)
dns_agg=dns.groupby(['device_id','date']).agg(
    dns_queries=('query_name','size'),max_dns_length=('query_length','max'),
    txt_queries=('query_type',lambda s:s.eq('TXT').sum()),
    unique_domains=('query_name','nunique')
).reset_index().rename(columns={'device_id':'terminal_id'})
features=features.merge(dns_agg,on=['terminal_id','date'],how='left',validate='one_to_one')
print(features[['terminal_id','date','external_flows','max_dns_length']].head())

              terminal_id       date  external_flows  max_dns_length
0  NRG-KHO-WDH-001-POS-01 2026-06-13               4              31
1  NRG-KHO-WDH-001-POS-01 2026-06-14               4              31
2  NRG-KHO-WDH-001-POS-01 2026-06-15               4              31
3  NRG-KHO-WDH-001-POS-01 2026-06-16               4              31
4  NRG-KHO-WDH-001-POS-01 2026-06-17               4              31


## 6. Access and endpoint evidence

The initial VPN failures and success are recorded on a branch administrative host. Use the shared session ID to attribute that VPN sequence to the first POS reached by Windows remote logon. Later remote logons originate from the previous POS and are counted as lateral access. Count unapproved processes and Windows service installs separately. Usernames are fictional, and used across related events.

In [48]:
auth=raw['auth'].copy(); auth['date']=auth.timestamp.dt.normalize();auth['hour']=auth.timestamp.dt.hour
auth=auth.merge(branches[['branch_id','trading_open','trading_close']],on='branch_id',validate='many_to_one')
auth['off_hours']=(auth.hour<auth.trading_open.str[:2].astype(int)) | (auth.hour>=auth.trading_close.str[:2].astype(int))
auth['lateral_login']=auth.auth_source.eq('windows_remote') & auth.source_device_id.fillna('').str.contains('-POS-')
auth_agg=auth.groupby(['device_id','date']).agg(
    remote_logins=('auth_source',lambda s:s.eq('windows_remote').sum()),
    lateral_logins=('lateral_login','sum'),
    off_hours_privileged=('off_hours',lambda s:int((s & auth.loc[s.index,'account_role'].eq('privileged_admin')).sum()))
).reset_index().rename(columns={'device_id':'terminal_id'})
# The VPN event belongs to ADM-01. Attribute it only to the POS reached
# through a Windows remote logon with the same session ID.
remote_targets=auth.loc[auth.auth_source.eq('windows_remote'),['session_id','device_id','date']].rename(columns={'device_id':'terminal_id'})
vpn_events=auth.loc[auth.auth_source.eq('vpn'),['session_id','result','username','source_ip']]
vpn_link=vpn_events.merge(remote_targets,on='session_id',how='inner',validate='many_to_one')
vpn_link['vpn_failed']=vpn_link.result.eq('failed').astype(int)
vpn_link['vpn_success']=vpn_link.result.eq('success').astype(int)
vpn_agg=vpn_link.groupby(['terminal_id','date'],as_index=False)[['vpn_failed','vpn_success']].sum()
features=features.merge(auth_agg,on=['terminal_id','date'],how='left',validate='one_to_one').merge(vpn_agg,on=['terminal_id','date'],how='left',validate='one_to_one')

edr=raw['edr'].copy();edr['date']=edr.timestamp.dt.normalize()
edr['unapproved']=~edr.process_name.isin(['VendorUpdater.exe','posapp.exe'])
edr_agg=edr.groupby(['host_id','date']).agg(unapproved_processes=('unapproved','sum')).reset_index().rename(columns={'host_id':'terminal_id'})
system=raw['system'].copy();system['date']=system.timestamp.dt.normalize()
system['event_id']=pd.to_numeric(system.event_id,errors='coerce')
sys_agg=system.groupby(['device_id','date']).agg(new_services=('event_id',lambda s:s.eq(7045).sum())).reset_index().rename(columns={'device_id':'terminal_id'})
features=features.merge(edr_agg,on=['terminal_id','date'],how='left',validate='one_to_one').merge(sys_agg,on=['terminal_id','date'],how='left',validate='one_to_one')

## 7. Complete and validate the feature table

There are no features from `ground_truth_private.csv`, disputes, ticket conclusions, event IDs or exact destination identifiers. Missing counts represent zero observed events. A monitoring gap may also produce zero events in real data; the synthetic data does not model outages, so production pipelines must add source-availability checks.

In [49]:
FEATURE_COLS=['sales_count','exception_count','override_count','total_amount',
 'external_flows','distinct_external_ips','bytes_out','bytes_in','max_flow_out',
 'off_hours_flows','long_sessions','out_in_ratio','dns_queries','max_dns_length',
 'txt_queries','unique_domains','vpn_failed','vpn_success','remote_logins','lateral_logins',
 'off_hours_privileged','unapproved_processes','new_services']
features[FEATURE_COLS]=features[FEATURE_COLS].apply(pd.to_numeric,errors='coerce').fillna(0)
assert len(features)==54*104 and not features.duplicated(['terminal_id','date']).any()
assert not features[FEATURE_COLS].isna().any().any()
features['exception_rate']=features.exception_count/(features.sales_count+1)
features['dow']=features.date.dt.dayofweek
features['weekend']=features.dow.ge(5).astype(int)
print('Feature shape:',features.shape)
display(features[FEATURE_COLS].describe().T[['mean','50%','max']])

Feature shape: (5616, 34)


,mean,50%,max
sales_count,9.498575,10.000000,11.00000
exception_count,0.161325,0.000000,1.00000
override_count,0.054665,0.000000,1.00000
total_amount,5321.118978,5302.725000,9212.84000
external_flows,4.290420,4.000000,11.00000
distinct_external_ips,2.038996,2.000000,3.00000
bytes_out,13963.086895,9528.000000,157984.00000
bytes_in,25535.854167,25281.500000,50587.00000
max_flow_out,4461.720085,3829.500000,26998.00000
off_hours_flows,0.174501,0.000000,4.00000


## 8. Learn clean baselines and reason codes

The first 14 days precede all six injected infections. Compute a median and median absolute deviation (MAD) per terminal and weekday/weekend, with peer-group fallback when a subgroup is empty. Since MAD is often zero for sparse event counts, use an absolute floor in each denominator. These baselines are fixed before scoring the analysis period and are not recomputed using later observations.

In [50]:
BASE_COLS=['bytes_out','out_in_ratio','off_hours_flows','max_dns_length',
           'txt_queries','vpn_failed','lateral_logins','unapproved_processes','new_services','exception_rate']
baseline=features.loc[features.date<ANALYSIS_START].copy()
med=baseline.groupby(['terminal_id','weekend'])[BASE_COLS].median()
mad=baseline.groupby(['terminal_id','weekend'])[BASE_COLS].agg(lambda s:np.median(np.abs(s-np.median(s))))
med.columns=[f'{c}_median' for c in BASE_COLS]
mad.columns=[f'{c}_mad' for c in BASE_COLS]
base_stats=med.join(mad).reset_index()
features=features.merge(base_stats,on=['terminal_id','weekend'],how='left',validate='many_to_one')
assert features[[f'{c}_median' for c in BASE_COLS]].notna().all().all()
FLOORS={'bytes_out':1000,'out_in_ratio':0.25,'off_hours_flows':1,'max_dns_length':5,
        'txt_queries':1,'vpn_failed':1,'lateral_logins':1,'unapproved_processes':1,'new_services':1,'exception_rate':0.05}
for c in BASE_COLS:
    features[c+'_z']=(features[c]-features[c+'_median'])/(1.4826*features[c+'_mad']+FLOORS[c])
features['baseline_score']=features[[c+'_z' for c in BASE_COLS]].clip(lower=0,upper=10).sum(axis=1)

def reasons(row):
    r=[]
    if row['bytes_out_z']>3 and row['out_in_ratio_z']>3:r.append('high outbound traffic with byte asymmetry')
    if row['off_hours_flows_z']>3:r.append('unexpected after-hours traffic')
    if row['max_dns_length_z']>3 or row['txt_queries_z']>3:r.append('unusual DNS queries')
    if row['vpn_failed']>=10 and row['vpn_success']>=1:r.append('ten failed VPN logins followed by success')
    if row['lateral_logins']>0:r.append('remote logon from another POS terminal')
    if row['off_hours_privileged']>0:r.append('privileged sign-in outside trading hours')
    if row['unapproved_processes']>0:r.append('unapproved POS process')
    if row['new_services']>0:r.append('new Windows service')
    return '; '.join(r) if r else 'No rule-level deviation'
features['reason_codes']=features.apply(reasons,axis=1)
display(features.loc[features.date>=ANALYSIS_START,['terminal_id','date','baseline_score','reason_codes']].sort_values('baseline_score',ascending=False).head(10))

,terminal_id,date,baseline_score,reason_codes
164,NRG-KHO-WDH-001-POS-02,2026-08-12,41.296077,high outbound traffic with byte asymmetry; unu...
167,NRG-KHO-WDH-001-POS-02,2026-08-15,40.200000,high outbound traffic with byte asymmetry; une...
4978,NRG-HAR-REH-011-POS-02,2026-09-11,39.422222,high outbound traffic with byte asymmetry; unu...
2698,NRG-OTJ-OTW-006-POS-01,2026-09-19,39.422222,high outbound traffic with byte asymmetry; unu...
803,NRG-ERG-WVB-002-POS-03,2026-08-27,39.422222,high outbound traffic with byte asymmetry; unu...
4966,NRG-HAR-REH-011-POS-02,2026-08-30,39.422222,high outbound traffic with byte asymmetry; unu...
3845,NRG-ZAM-KTM-008-POS-03,2026-09-22,39.422222,high outbound traffic with byte asymmetry; unu...
204,NRG-KHO-WDH-001-POS-02,2026-09-21,39.422222,high outbound traffic with byte asymmetry; unu...
202,NRG-KHO-WDH-001-POS-02,2026-09-19,39.422222,high outbound traffic with byte asymmetry; unu...
2677,NRG-OTJ-OTW-006-POS-01,2026-08-29,39.422222,high outbound traffic with byte asymmetry; unu...


## 9. Unsupervised anomaly detector

Fit Isolation Forest only on clean baseline terminal-days, then score the 90-day analysis. Larger `anomaly_score` means more unusual. The top 5% of baseline scores defines an alert budget; we later report its realised false-positive rate on holdout days.

In [51]:
MODEL_COLS=FEATURE_COLS+['exception_rate','weekend']
imputer=SimpleImputer(strategy='median').fit(baseline[MODEL_COLS])
X_all=imputer.transform(features[MODEL_COLS])
X_base=imputer.transform(baseline[MODEL_COLS])
iso=IsolationForest(n_estimators=200,random_state=82119,contamination='auto',n_jobs=-1)
iso.fit(X_base)
features['anomaly_score']=-iso.decision_function(X_all)
base_score=-iso.decision_function(X_base)
features['anomaly_alert']=features.anomaly_score>np.quantile(base_score,0.95)
display(features.loc[(features.date>=ANALYSIS_START)&features.anomaly_alert,
 ['terminal_id','date','anomaly_score','reason_codes']].sort_values('anomaly_score',ascending=False).head(10))


# Troubleshooting
#print("Data folder:", DATA_DIR)

#print("\nGround-truth files in this folder:")
#for path in DATA_DIR.glob('ground_truth*'):
#    print(path.name)

#truth = pd.read_csv(DATA_DIR / 'ground_truth_private.csv')
#truth['terminal_id'] = truth['terminal_id'].astype(str).str.strip()
#truth['compromise_start'] = pd.to_datetime(
#    truth['compromise_start'], errors='raise'
#)

#features['terminal_id'] = features['terminal_id'].astype(str).str.strip()
#features['date'] = pd.to_datetime(features['date'], errors='raise')

#print("\nFeature terminal IDs:")
#print(features['terminal_id'].drop_duplicates().head(10).tolist())

#print("\nGround-truth terminal IDs and dates:")
#print(truth[['terminal_id', 'compromise_start']].to_string(index=False))

#matching = set(features['terminal_id']) & set(truth['terminal_id'])
#print("\nMatching compromised terminals:", len(matching))

#print("\nFeature date range:")
#print(features['date'].min(), "to", features['date'].max())


,terminal_id,date,anomaly_score,reason_codes
1756,NRG-OSA-OSK-004-POS-02,2026-09-13,0.244234,high outbound traffic with byte asymmetry; unu...
3828,NRG-ZAM-KTM-008-POS-03,2026-09-05,0.238164,high outbound traffic with byte asymmetry; une...
1744,NRG-OSA-OSK-004-POS-02,2026-09-01,0.236756,high outbound traffic with byte asymmetry; unu...
1763,NRG-OSA-OSK-004-POS-02,2026-09-20,0.230020,high outbound traffic with byte asymmetry; unu...
4977,NRG-HAR-REH-011-POS-02,2026-09-10,0.228931,high outbound traffic with byte asymmetry; une...
183,NRG-KHO-WDH-001-POS-02,2026-08-31,0.228793,high outbound traffic with byte asymmetry; unu...
202,NRG-KHO-WDH-001-POS-02,2026-09-19,0.228591,high outbound traffic with byte asymmetry; unu...
4991,NRG-HAR-REH-011-POS-02,2026-09-24,0.228142,high outbound traffic with byte asymmetry; une...
3845,NRG-ZAM-KTM-008-POS-03,2026-09-22,0.226679,high outbound traffic with byte asymmetry; unu...
2698,NRG-OTJ-OTW-006-POS-01,2026-09-19,0.224433,high outbound traffic with byte asymmetry; unu...


## 10. Supervised detector and time-held-out evaluation

The private label file defines the start of each simulated compromise. Label a terminal-day compromised when its date is on or after that start. Train on days through 31 August, choose a threshold from **1–10 September** validation negatives, and report metrics on **11–24 September**, which were not used for fitting or threshold choice. The dates matter: a random row split would put days from the same terminal and incident in both train and test. Even this chronological split tests continuation of the same six incidents, not generalisation to unseen malware campaigns. Use labels only inside this evaluation cell and never feed the label or infection date into `MODEL_COLS`.

In [52]:
truth=pd.read_csv(DATA_DIR/'ground_truth_private.csv',parse_dates=['compromise_start'])
first_day=truth.set_index('terminal_id').compromise_start.dt.normalize()
features['label']=((features.terminal_id.map(first_day).notna()) &
                   (features.date>=features.terminal_id.map(first_day))).astype(int)
train=(features.date>=ANALYSIS_START)&(features.date<pd.Timestamp('2026-09-01'))
valid=(features.date>=pd.Timestamp('2026-09-01'))&(features.date<pd.Timestamp('2026-09-11'))
test=(features.date>=pd.Timestamp('2026-09-11'))&(features.date<END_EXCLUSIVE)
print('Train/validation/test positives:',*(int(features.loc[m,'label'].sum()) for m in (train,valid,test)))
assert features.loc[train,'label'].nunique()==2 and features.loc[valid,'label'].nunique()==2 and features.loc[test,'label'].nunique()==2
model=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=15,max_depth=4,
                                     learning_rate=0.05,l2_regularization=1,random_state=82119,
                                     early_stopping=False)
model.fit(X_all[train.to_numpy()],features.loc[train,'label'])
features['model_probability']=model.predict_proba(X_all)[:,1]
neg_valid=features.loc[valid & features.label.eq(0),'model_probability'].to_numpy()
# Strict '>' on the 95th percentile targets at most about 5% validation false positives.
threshold=float(np.quantile(neg_valid,0.95))
features['model_alert']=features.model_probability>threshold

def evaluate(mask,col):
    y=features.loc[mask,'label'].to_numpy(); pred=features.loc[mask,col].astype(int).to_numpy()
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'TP':tp,'FP':fp,'FN':fn,'TN':tn,
            'precision':precision_score(y,pred,zero_division=0),
            'recall':recall_score(y,pred,zero_division=0),
            'F1':f1_score(y,pred,zero_division=0),
            'FPR':fp/(fp+tn) if fp+tn else np.nan}
print('Threshold:',threshold)
print('Validation model:',evaluate(valid,'model_alert'))
print('Held-out model:',evaluate(test,'model_alert'))
print('Held-out anomaly:',evaluate(test,'anomaly_alert'))
print('Held-out baseline rule:',evaluate(test,'baseline_score_alert') if 'baseline_score_alert' in features else 'See next cell')

Train/validation/test positives: 75 60 84
Threshold: 0.0003292170530219975
Validation model: {'TP': np.int64(60), 'FP': np.int64(0), 'FN': np.int64(0), 'TN': np.int64(480), 'precision': 1.0, 'recall': 1.0, 'F1': 1.0, 'FPR': np.float64(0.0)}
Held-out model: {'TP': np.int64(84), 'FP': np.int64(0), 'FN': np.int64(0), 'TN': np.int64(672), 'precision': 1.0, 'recall': 1.0, 'F1': 1.0, 'FPR': np.float64(0.0)}
Held-out anomaly: {'TP': np.int64(84), 'FP': np.int64(25), 'FN': np.int64(0), 'TN': np.int64(647), 'precision': 0.7706422018348624, 'recall': 1.0, 'F1': 0.8704663212435233, 'FPR': np.float64(0.03720238095238095)}
Held-out baseline rule: See next cell


## 11. Compare the baseline rule and rank terminals

Set a rule threshold using only the clean baseline score distribution. Show the test period comparison, then rank terminal-days using model probability, anomaly score and reason codes. The model probability is a score from this synthetic training distribution, not a calibrated real-world probability of compromise.

In [53]:
rule_cut=float(np.quantile(features.loc[features.date<ANALYSIS_START,'baseline_score'],0.95))
features['baseline_score_alert']=features.baseline_score>rule_cut
print('Held-out baseline rule:',evaluate(test,'baseline_score_alert'))
ranked=features.loc[features.date>=ANALYSIS_START,
 ['branch_id','region','terminal_id','date','model_probability','anomaly_score',
  'baseline_score','model_alert','anomaly_alert','reason_codes']].sort_values(
 ['model_probability','anomaly_score'],ascending=False)
display(ranked.head(20))
ranked.to_csv(OUT/'ranked_terminal_days.csv',index=False)
features.drop(columns=['label']).to_csv(OUT/'terminal_day_features.csv',index=False)

Held-out baseline rule: {'TP': np.int64(84), 'FP': np.int64(40), 'FN': np.int64(0), 'TN': np.int64(632), 'precision': 0.6774193548387096, 'recall': 1.0, 'F1': 0.8076923076923077, 'FPR': np.float64(0.05952380952380952)}


,branch_id,region,terminal_id,date,model_probability,anomaly_score,baseline_score,model_alert,anomaly_alert,reason_codes
1756,NRG-OSA-OSK-004,Oshana,NRG-OSA-OSK-004-POS-02,2026-09-13,0.984618,0.244234,39.018182,True,True,high outbound traffic with byte asymmetry; unu...
3828,NRG-ZAM-KTM-008,Zambezi,NRG-ZAM-KTM-008-POS-03,2026-09-05,0.984618,0.238164,39.228190,True,True,high outbound traffic with byte asymmetry; une...
1744,NRG-OSA-OSK-004,Oshana,NRG-OSA-OSK-004-POS-02,2026-09-01,0.984618,0.236756,38.866667,True,True,high outbound traffic with byte asymmetry; unu...
1763,NRG-OSA-OSK-004,Oshana,NRG-OSA-OSK-004-POS-02,2026-09-20,0.984618,0.230020,38.866667,True,True,high outbound traffic with byte asymmetry; unu...
4977,NRG-HAR-REH-011,Hardap,NRG-HAR-REH-011-POS-02,2026-09-10,0.984618,0.228931,36.880905,True,True,high outbound traffic with byte asymmetry; une...
183,NRG-KHO-WDH-001,Khomas,NRG-KHO-WDH-001-POS-02,2026-08-31,0.984618,0.228793,37.764016,True,True,high outbound traffic with byte asymmetry; unu...
202,NRG-KHO-WDH-001,Khomas,NRG-KHO-WDH-001-POS-02,2026-09-19,0.984618,0.228591,39.422222,True,True,high outbound traffic with byte asymmetry; unu...
4991,NRG-HAR-REH-011,Hardap,NRG-HAR-REH-011-POS-02,2026-09-24,0.984618,0.228142,37.060350,True,True,high outbound traffic with byte asymmetry; une...
3845,NRG-ZAM-KTM-008,Zambezi,NRG-ZAM-KTM-008-POS-03,2026-09-22,0.984618,0.226679,39.422222,True,True,high outbound traffic with byte asymmetry; unu...
2698,NRG-OTJ-OTW-006,Otjozondjupa,NRG-OTJ-OTW-006-POS-01,2026-09-19,0.984618,0.224433,39.422222,True,True,high outbound traffic with byte asymmetry; unu...


## 12. Investigate the VPN entry point and lateral movement

Select flagged terminals and join remote logons by `session_id`, `source_device_id` and target POS. The first terminal's session connects ten VPN failures, a VPN success and Windows remote access. The other five terminals have remote logons from a previously compromised POS and TCP/445 flows to the target address. Compare these against EDR, DNS and outward flows. Disputes are retrospective evidence and are never included in the detector. All named accounts and domains are invented scenario data.

In [54]:
flagged=features.loc[test & features.model_alert,'terminal_id'].drop_duplicates().tolist()
if not flagged: flagged=ranked.terminal_id.drop_duplicates().head(6).tolist()
print('Terminals selected:',flagged)

investigation=[]
for t in flagged:
    d=devices.loc[devices.device_id.eq(t)].iloc[0]
    e=edr.loc[edr.host_id.eq(t)].sort_values('timestamp')
    suspicious_e=e.loc[~e.process_name.isin(['VendorUpdater.exe','posapp.exe'])]
    n=extflows.loc[extflows.src_device_id.eq(t)].sort_values('timestamp')
    suspicious_n=n.loc[(n.bytes_out>10000)&(n.bytes_out>3*n.bytes_in)]
    q=dns.loc[dns.device_id.eq(t)].sort_values('timestamp')
    suspicious_q=q.loc[(q.query_length>55)|q.query_type.eq('TXT')]
    remote=auth.loc[auth.device_id.eq(t)&auth.auth_source.eq('windows_remote')].sort_values('timestamp')
    session_ids=set(remote.session_id)
    vpn=auth.loc[auth.auth_source.eq('vpn')&auth.session_id.isin(session_ids)].sort_values('timestamp')
    lateral=flows.loc[flows.dst_ip.eq(d.ip_address)&flows.dst_port.eq(445)]
    payments=raw['payments'].loc[raw['payments'].terminal_id.eq(t)]
    times=[suspicious_e.timestamp.min(),suspicious_n.timestamp.min(),suspicious_q.timestamp.min(),
           remote.timestamp.min(),vpn.timestamp.min(),lateral.timestamp.min()]
    first_seen=min((v for v in times if pd.notna(v)),default=pd.NaT)
    investigation.append({'terminal_id':t,'branch_id':d.branch_id,
        'first_observed_attack_signal':first_seen,
        'first_vpn_attempt':vpn.timestamp.min(),
        'vpn_failures':int(vpn.result.eq('failed').sum()),
        'vpn_successes':int((vpn.result.eq('success')).sum()),
        'vpn_username':vpn.username.iloc[0] if len(vpn) else '',
        'remote_username':remote.username.iloc[0] if len(remote) else '',
        'parent_device':remote.source_device_id.iloc[0] if len(remote) else '',
        'first_remote_logon':remote.timestamp.min(),
        'first_lateral_smb':lateral.timestamp.min(),
        'first_unapproved_process':suspicious_e.timestamp.min(),
        'first_asymmetric_flow':suspicious_n.timestamp.min(),
        'first_unusual_dns':suspicious_q.timestamp.min(),
        'suspicious_outbound_bytes':int(suspicious_n.bytes_out.sum()),
        'transactions_at_terminal':len(payments),
        'later_disputes':int(pd.to_numeric(payments.dispute_flag,errors='coerce').fillna(0).sum())})
incident_summary=pd.DataFrame(investigation).sort_values('first_observed_attack_signal')
display(incident_summary)
incident_summary.to_csv(OUT/'incident_summary.csv',index=False)

# One event-level timeline. The first flagged terminal is usually the VPN entry point.
selected=incident_summary.iloc[0].terminal_id
selected_ip=devices.loc[devices.device_id.eq(selected),'ip_address'].iloc[0]
remote=auth.loc[auth.device_id.eq(selected)&auth.auth_source.eq('windows_remote')]
sessions=set(remote.session_id)
timeline=pd.concat([
    auth.loc[auth.session_id.isin(sessions)&auth.auth_source.isin(['vpn','windows_remote']),
             ['timestamp','auth_source','username','result']].assign(source='Authentication',
             detail=lambda x:x.auth_source+' '+x.username+' '+x.result)[['timestamp','source','detail']],
    flows.loc[flows.dst_ip.eq(selected_ip)&flows.dst_port.eq(445),['timestamp','src_device_id']]
         .rename(columns={'src_device_id':'detail'}).assign(source='Lateral SMB'),
    edr.loc[edr.host_id.eq(selected),['timestamp','process_name']]
         .rename(columns={'process_name':'detail'}).assign(source='EDR'),
    extflows.loc[extflows.src_device_id.eq(selected)&(extflows.bytes_out>10000),['timestamp','url_host']]
         .rename(columns={'url_host':'detail'}).assign(source='Network'),
    dns.loc[dns.device_id.eq(selected)&(dns.query_length>55),['timestamp','query_name']]
         .rename(columns={'query_name':'detail'}).assign(source='DNS')
],ignore_index=True).sort_values('timestamp')
print('Attack timeline for',selected)
display(timeline.head(35))
timeline.to_csv(OUT/'selected_terminal_timeline.csv',index=False)

Terminals selected: ['NRG-KHO-WDH-001-POS-02', 'NRG-ERG-WVB-002-POS-03', 'NRG-OSA-OSK-004-POS-02', 'NRG-OTJ-OTW-006-POS-01', 'NRG-ZAM-KTM-008-POS-03', 'NRG-HAR-REH-011-POS-02']


,terminal_id,branch_id,first_observed_attack_signal,first_vpn_attempt,vpn_failures,vpn_successes,vpn_username,remote_username,parent_device,first_remote_logon,first_lateral_smb,first_unapproved_process,first_asymmetric_flow,first_unusual_dns,suspicious_outbound_bytes,transactions_at_terminal,later_disputes
0,NRG-KHO-WDH-001-POS-02,NRG-KHO-WDH-001,2026-08-12 08:40:00,2026-08-12 08:40:00,10,1,johannes.muutota,johannes.muutota,NRG-KHO-WDH-001-ADM-01,2026-08-12 09:04:00,NaT,2026-08-12 09:15:00,2026-08-12 13:02:00,2026-08-12 13:02:01,5067975,979,5
1,NRG-ERG-WVB-002-POS-03,NRG-ERG-WVB-002,2026-08-15 09:03:00,NaT,0,0,,johannes.muutota,NRG-KHO-WDH-001-POS-02,2026-08-15 09:03:00,2026-08-15 09:03:00,2026-08-15 09:15:00,2026-08-15 13:03:00,2026-08-15 13:03:01,4907038,1010,5
2,NRG-OSA-OSK-004-POS-02,NRG-OSA-OSK-004,2026-08-18 09:03:00,NaT,0,0,,johannes.muutota,NRG-ERG-WVB-002-POS-03,2026-08-18 09:03:00,2026-08-18 09:03:00,2026-08-18 09:15:00,2026-08-18 13:01:00,2026-08-18 13:01:01,4439303,1009,2
3,NRG-OTJ-OTW-006-POS-01,NRG-OTJ-OTW-006,2026-08-21 09:03:00,NaT,0,0,,johannes.muutota,NRG-OSA-OSK-004-POS-02,2026-08-21 09:03:00,2026-08-21 09:03:00,2026-08-21 09:15:00,2026-08-21 13:00:00,2026-08-21 13:00:01,4012664,964,6
4,NRG-ZAM-KTM-008-POS-03,NRG-ZAM-KTM-008,2026-08-24 09:03:00,NaT,0,0,,johannes.muutota,NRG-OTJ-OTW-006-POS-01,2026-08-24 09:03:00,2026-08-24 09:03:00,2026-08-24 09:15:00,2026-08-24 13:04:00,2026-08-24 13:04:01,3718250,991,6
5,NRG-HAR-REH-011-POS-02,NRG-HAR-REH-011,2026-08-27 09:03:00,NaT,0,0,,johannes.muutota,NRG-ZAM-KTM-008-POS-03,2026-08-27 09:03:00,2026-08-27 09:03:00,2026-08-27 09:15:00,2026-08-27 13:04:00,2026-08-27 13:04:01,3457071,969,3


Attack timeline for NRG-KHO-WDH-001-POS-02


,timestamp,source,detail
12,2026-07-03 10:00:00,EDR,VendorUpdater.exe
13,2026-08-03 10:00:00,EDR,VendorUpdater.exe
0,2026-08-12 08:40:00,Authentication,vpn johannes.muutota failed
1,2026-08-12 08:42:00,Authentication,vpn johannes.muutota failed
2,2026-08-12 08:44:00,Authentication,vpn johannes.muutota failed
3,2026-08-12 08:46:00,Authentication,vpn johannes.muutota failed
4,2026-08-12 08:48:00,Authentication,vpn johannes.muutota failed
5,2026-08-12 08:50:00,Authentication,vpn johannes.muutota failed
6,2026-08-12 08:52:00,Authentication,vpn johannes.muutota failed
7,2026-08-12 08:54:00,Authentication,vpn johannes.muutota failed


## 13. Save a short run record

Record the data window, test metrics and output paths in a text file. Do not include the private ground truth in a dashboard or operational export.

In [55]:
summary={
 'rows':len(features),'baseline_days':14,'analysis_days':90,
 'validation_threshold':threshold,'rule_threshold':rule_cut,
 'held_out_model':evaluate(test,'model_alert'),
 'held_out_anomaly':evaluate(test,'anomaly_alert'),
 'held_out_rule':evaluate(test,'baseline_score_alert')
}
import json
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2,default=float))
print('Saved:',*[str(x) for x in OUT.iterdir()],sep='\n- ')

Saved:
- /content/drive/MyDrive/CardWatch/outputs/ranked_terminal_days.csv
- /content/drive/MyDrive/CardWatch/outputs/terminal_day_features.csv
- /content/drive/MyDrive/CardWatch/outputs/incident_summary.csv
- /content/drive/MyDrive/CardWatch/outputs/selected_terminal_timeline.csv
- /content/drive/MyDrive/CardWatch/outputs/run_summary.json
